# Y-->Z axil Transform

In [ ]:
# -*- coding: utf-8 -*-
import os

import numpy as np


# Input/output directories
in_dir = "./data/Data_release_Trumans/Scene"
out_dir = "./data/Trumans_Scene_Processed/npy_z_up"
os.makedirs(out_dir, exist_ok=True)


def yup_to_zup_voxel(voxel: np.ndarray):
    """
    Convert Y-up coordinates (x, y, z) to Z-up coordinates (x, z, -y).

    This is equivalent to a -90-degree rotation around the x-axis.
    """
    assert voxel.ndim == 3

    voxel_tmp = np.transpose(voxel, (0, 2, 1))

    voxel_zup = np.flip(voxel_tmp, axis=2)

    return voxel_zup


# Process all .npy files
files = [f for f in os.listdir(in_dir) if f.endswith(".npy")]
print(f"Found {len(files)} voxel files")

for i, fname in enumerate(files, 1):
    in_path = os.path.join(in_dir, fname)
    out_path = os.path.join(out_dir, fname)

    data = np.load(in_path)  # Expected shape: (300, 100, 400)
    assert data.ndim == 3, f"{fname} is not a 3D voxel array: {data.shape}"

    data_zup = yup_to_zup_voxel(data)
    np.save(out_path, data_zup)

    if i % 20 == 0 or i == len(files):
        print(
            f"[{i}/{len(files)}] Processed {fname}, "
            f"shape {data.shape} -> {data_zup.shape}"
        )

print("Conversion completed. Output directory:", out_dir)

# Remove Floor

In [ ]:
# -*- coding: utf-8 -*-
import os
import numpy as np

def remove_floor(voxel_file, out_file=None, min_ratio=0.6, max_check_layers=50):

    occ = np.load(voxel_file)
    occ = occ.astype(bool)
    
    floor_layers = [0]
    occ_no_floor = occ.copy()
    for z in floor_layers:
        occ_no_floor[:, :, z] = False

    if out_file is not None:
        os.makedirs(os.path.dirname(out_file), exist_ok=True)
        np.save(out_file, occ_no_floor)

    return occ_no_floor, floor_layers


if __name__ == "__main__":
    in_dir  = "./data/Trumans_Scene_Processed/npy_z_up"           
    out_dir = "./data/Trumans_Scene_Processed/npy_z_up_wo_floor"  
    os.makedirs(out_dir, exist_ok=True)

    files = [f for f in os.listdir(in_dir) if f.endswith(".npy")]

    for i, fname in enumerate(files, 1):
        in_path  = os.path.join(in_dir, fname)
        out_path = os.path.join(out_dir, fname)

        occ_no_floor, floor_layers = remove_floor(in_path, out_path)


    print("🎉 out_dir:", out_dir)


# Collision Mesh Generation

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import time
import numpy as np
import open3d as o3d
from skimage import measure

from scipy.ndimage import distance_transform_edt, gaussian_filter, binary_closing



IN_DIR   = "./data/Trumans_Scene_Processed/npy_z_up_wo_floor"
OUT_ROOT = "./data/Trumans_Scene_Processed/collision_mesh"


TARGET_FILES = None


os.makedirs(OUT_ROOT, exist_ok=True)



WORLD_SIZE = np.array([6.0, 8.0, 2.0], dtype=np.float32)  
OFFSET     = np.array([-3.0, -4.0, 0.0], dtype=np.float32) 



DOWNSAMPLE_FACTOR   = 2     
MC_STEP             = 1      
MAX_TRIS_PER_MESH   = 150_000  

WRITE_VERTEX_NORMALS = True


USE_SDF             = True
SDF_SMOOTH_SIGMA    = 0.0    
MORPH_CLOSING_ITERS = 1     
TAUBIN_ITERS        = 5     


# ======================================

def as_float32(a):
    return np.asarray(a, dtype=np.float32, order="C")


def downsample_maxpool(voxel: np.ndarray, factor: int):
    if factor <= 1:
        return voxel, False

    D, H, W = voxel.shape

    d2, h2, w2 = D // factor, H // factor, W // factor
    v = voxel.reshape(d2, factor, h2, factor, w2, factor)
    v_ds = v.max(axis=(1, 3, 5))  
    return v_ds.astype(voxel.dtype), True


def build_sdf(occ: np.ndarray,
              smooth_sigma: float = 0.0,
              closing_iters: int = 0) -> np.ndarray:
    occ = (occ > 0)

    if closing_iters > 0:
        occ = binary_closing(occ, iterations=int(closing_iters))

    if occ.sum() == 0:
        return np.zeros_like(occ, dtype=np.float32)

    inside = occ
    outside = ~occ

    dist_inside = distance_transform_edt(inside)
    dist_outside = distance_transform_edt(outside)

    sdf = dist_inside - dist_outside

    if smooth_sigma > 0:
        sdf = gaussian_filter(sdf, sigma=float(smooth_sigma))

    return sdf.astype(np.float32)


def repair_mesh_topology(mesh: o3d.geometry.TriangleMesh):
    mesh.remove_degenerate_triangles()
    mesh.remove_duplicated_triangles()
    mesh.remove_duplicated_vertices()
    mesh.remove_non_manifold_edges()
    mesh.remove_unreferenced_vertices()


def export_single_mesh(mesh,
                       base_name,
                       out_dir,
                       taubin_iters=TAUBIN_ITERS,
                       write_normals=WRITE_VERTEX_NORMALS,
                       max_tris=MAX_TRIS_PER_MESH):
    
    if len(mesh.triangles) == 0:
        return 0

    m = mesh
    if taubin_iters > 0:
        m = m.filter_smooth_taubin(number_of_iterations=int(taubin_iters))

    n_tris = np.asarray(m.triangles).shape[0]
    if max_tris is not None and n_tris > max_tris:
        target = int(max_tris)
        m = m.simplify_quadric_decimation(target_number_of_triangles=target)
        print(f"  [Decimate] tris {n_tris} -> {len(m.triangles)}")

    m.compute_vertex_normals()
    out_file = os.path.join(out_dir, f"{base_name}.obj")
    o3d.io.write_triangle_mesh(out_file, m, write_vertex_normals=write_normals)
    print(f"  [Save] {out_file}")
    return 1


def main():
    all_files = [f for f in os.listdir(IN_DIR) if f.endswith(".npy")]
    all_files.sort()

    if TARGET_FILES is not None:
        files = [f for f in all_files if f in TARGET_FILES]
        print(f"[INFO] Processing target scene only: {files}")
    else:
        files = all_files
        print(f"[INFO] Processing ALL {len(files)} voxel files")

    print(f"[INFO] Output root: {OUT_ROOT}")
    print(f"[INFO] DOWNSAMPLE_FACTOR={DOWNSAMPLE_FACTOR}, MC_STEP={MC_STEP}, USE_SDF={USE_SDF}")

    for idx, fname in enumerate(files, 1):
        t0 = time.time()
        in_path = os.path.join(IN_DIR, fname)
        base = os.path.splitext(fname)[0]

        out_dir = os.path.join(OUT_ROOT, base)
        os.makedirs(out_dir, exist_ok=True)

        occ_raw = np.load(in_path).astype(np.uint8, copy=False)
        D, H, W = occ_raw.shape
        occ_raw_nz = int(occ_raw.sum())

        print(f"\n[{idx}/{len(files)}] {base}")
        print(f"  raw shape = {occ_raw.shape}, raw occupied = {occ_raw_nz}")

        occ_ds, _ = downsample_maxpool(occ_raw, DOWNSAMPLE_FACTOR)
        D2, H2, W2 = occ_ds.shape
        print(f"  downsampled shape = {occ_ds.shape}, occupied = {int(occ_ds.sum())}")

        if occ_ds.sum() == 0:
            print("  [Skip] no occupied voxels after downsample.")
            continue

        if USE_SDF:
            field = build_sdf(
                occ_ds,
                smooth_sigma=SDF_SMOOTH_SIGMA,
                closing_iters=MORPH_CLOSING_ITERS
            )
            level = 0.0
            field_type = "SDF"
        else:
            field = occ_ds.astype(np.float32)
            level = 0.5
            field_type = "binary"

        print(f"  field type = {field_type}, min={field.min():.3f}, max={field.max():.3f}")

        # 3) marching cubes
        spacing = WORLD_SIZE / np.array([D2, H2, W2], dtype=np.float32)

        try:
            verts, faces, _, _ = measure.marching_cubes(
                volume=field,
                level=float(level),
                spacing=tuple(map(float, spacing)),
                step_size=MC_STEP,
                allow_degenerate=False,
                gradient_direction='ascent',
            )
        except RuntimeError as e:
            print(f"  [Error] marching_cubes failed: {e}")
            continue

        if verts.size == 0 or faces.size == 0:
            print("  [Skip] no vertices or faces from marching_cubes.")
            continue

        verts_world = as_float32(verts) + as_float32(OFFSET)

        mesh = o3d.geometry.TriangleMesh()
        mesh.vertices  = o3d.utility.Vector3dVector(verts_world.astype(np.float64))
        mesh.triangles = o3d.utility.Vector3iVector(faces.astype(np.int32))

        repair_mesh_topology(mesh)

        n_tris_before = np.asarray(mesh.triangles).shape[0]
        exported = export_single_mesh(
            mesh,
            base_name=base,
            out_dir=out_dir,
            taubin_iters=TAUBIN_ITERS,
            write_normals=WRITE_VERTEX_NORMALS,
            max_tris=MAX_TRIS_PER_MESH
        )

        bbox = mesh.get_axis_aligned_bounding_box()
        print("  [BBox] min:", np.round(bbox.min_bound, 4),
              "max:", np.round(bbox.max_bound, 4))
        print(f"  [Info] tris (before decimation) = {n_tris_before}, exported {exported} meshes")
        print(f"  [Time] {time.time() - t0:.2f}s")

    print("\n✅ Done.")


if __name__ == "__main__":
    main()
